# 01 — TiNA Data Understanding

**Goal:** verify the source schema, time coverage, missingness, and maintenance/alarm distributions before preprocessing.

The official source is [ari-dasci/OD-TINA](https://github.com/ari-dasci/OD-TINA). The dataset is too large to load eagerly on a typical laptop, so this notebook starts with a bounded preview. Do not commit source data or notebook outputs containing large tables.

## Reproducibility record

Complete this before sharing results:

- Source commit/download date: **TODO**
- Local filename/checksum: **TODO**
- Selected period and rationale: **TODO**
- Analyst: **TODO**

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import resolve_data_path, scan_tina, validate_event_columns

DATA_PATH = resolve_data_path()
PREVIEW_ROWS = int(os.environ.get('TINA_PREVIEW_ROWS', '250000'))
print(f'Data path: {DATA_PATH}')
print(f'Bounded preview rows: {PREVIEW_ROWS:,}')

## Load lazily and validate

The preview supports quick iteration. Final claims must be rerun against a documented, event-preserving time period rather than an arbitrary row sample.

In [ ]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
schema = lazy.collect_schema()
print(f'Columns: {len(schema.names())}')
schema

In [ ]:
preview = lazy.head(PREVIEW_ROWS).collect()
print(f'Preview shape: {preview.shape}')
preview.head()

## Time field and ordering

Confirm the real timestamp column manually if it is not detected. Never build lags, rolling features, or train/test splits until ascending time order is verified.

In [ ]:
TIME_CANDIDATES = ('timestamp', 'datetime', 'date', 'time', 'Date', 'Time')
time_col = next((name for name in TIME_CANDIDATES if name in preview.columns), None)
print(f'Detected time column: {time_col!r}')
if time_col:
    print(preview.select(pl.col(time_col).min().alias('min_time'), pl.col(time_col).max().alias('max_time')))
    print(f'Ascending in preview: {preview[time_col].is_sorted()}')
else:
    print('Set time_col to the verified timestamp field before continuing to time-based analysis.')

## Missingness and constant fields

In [ ]:
quality = pl.DataFrame({
    'column': preview.columns,
    'dtype': [str(dtype) for dtype in preview.dtypes],
    'null_count': [preview[column].null_count() for column in preview.columns],
    'n_unique': [preview[column].n_unique() for column in preview.columns],
}).with_columns((pl.col('null_count') / preview.height * 100).alias('null_pct'))
quality.sort('null_pct', descending=True).head(25)

## Maintenance and alarm distributions

In [ ]:
for column in ('m_id', 'm_subid', 'alarms'):
    display(
        preview.group_by(column)
        .len(name='rows')
        .with_columns((pl.col('rows') / preview.height * 100).alias('pct'))
        .sort('rows', descending=True)
        .head(20)
    )

## Initial decisions

Record findings before moving to notebook 02:

- Verified timestamp and sampling interval: **TODO**
- Data-quality issues requiring treatment: **TODO**
- Maintenance/alarm codes present in the selected period: **TODO**
- Event-preserving subset and memory estimate: **TODO**
- Candidate prediction horizon(s): **TODO**
- Known recovery periods or rows to exclude: **TODO**